# 🕸️ LangGraph Complete Tutorial
## Building Stateful, Cyclical AI Workflows

---

### What You Will Learn
| Section | Topic |
|---|---|
| 1 | Why LangGraph exists — LangChain's limitation |
| 2 | Core concepts: State, Nodes, Edges |
| 3 | Your first graph — linear flow |
| 4 | Conditional edges — branching (if/else) |
| 5 | Loops — cycles that iterate until done |
| 6 | Persistence — save and resume graph state |
| 7 | Multi-agent — supervisor coordinates specialists |
| 8 | Human-in-the-loop — pause for human approval |
| 9 | Real projects |

---

### LangChain vs LangGraph

```
LangChain Chain:          LangGraph:

A → B → C → Done          A → B → C
                               ↑   │ (conditional)
(linear, no loops)             └───┘ (loop back)
                              or → D → Done
```

| Feature | LangChain | LangGraph |
|---|---|---|
| **Structure** | Linear chain | Graph (nodes + edges) |
| **State** | No built-in state | Typed state passed between nodes |
| **Loops** | Not supported | First-class feature |
| **Branching** | Manual workaround | Conditional edges |
| **Human approval** | Complex workaround | Built-in interrupt |
| **Multi-agent** | Complex to wire | Natural with supervisor pattern |

**Java analogy:** LangChain chains = method chaining (builder pattern).
LangGraph = a state machine / workflow engine (like a BPM or Apache Airflow DAG
but with cycles allowed).

In [ ]:
# Install LangGraph and dependencies
!pip install langgraph langchain langchain-openai -q

In [ ]:
from dotenv import load_dotenv
import os

# override=True ensures the latest key from .env is always used,
# even if the kernel already has an old key cached in memory
load_dotenv(override=True)

print("Key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

---
## Section 2 — Core Concepts

### 1. State (TypedDict)
A typed dictionary shared between ALL nodes in the graph.
Every node reads from it and can write updates back to it.
```python
class MyState(TypedDict):
    user_input: str       # set at the start
    sentiment: str        # filled by node 1
    response: str         # filled by node 2
```
**Java analogy:** A shared `Map<String, Object>` that all processing steps read/write.

### 2. Nodes
Regular Python functions. Each node:
- Receives the current state as input
- Returns a **dict of updates** (only the fields that changed)
```python
def my_node(state: MyState) -> dict:
    result = do_something(state["user_input"])
    return {"sentiment": result}  # only return what changed
```

### 3. Edges
- **Normal edge:** `graph.add_edge("A", "B")` — always go from A to B
- **Conditional edge:** call a function to decide which node to go to next
- **END:** special terminal node — graph stops here

### 4. Building a Graph
```python
graph = StateGraph(MyState)     # 1. create graph
graph.add_node("name", fn)      # 2. add nodes
graph.set_entry_point("first")  # 3. set start
graph.add_edge("A", "B")        # 4. connect nodes
app = graph.compile()           # 5. compile
result = app.invoke({...})      # 6. run
```

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o", temperature=0.3)

# ============================================================
# Your First LangGraph: Sentiment → Response
# Flow: analyze_sentiment → generate_response → END
# ============================================================

# --- Step 1: Define State ---
# Everything that flows through the graph lives here
class CustomerState(TypedDict):
    message: str      # customer's input — set at start
    sentiment: str    # filled by analyze_sentiment node
    response: str     # filled by generate_response node

# --- Step 2: Define Nodes ---

def analyze_sentiment(state: CustomerState) -> dict:
    """Node 1: Determine sentiment of the customer message."""
    result = llm.invoke([
        HumanMessage(content=(
            f"Classify as POSITIVE, NEGATIVE, or NEUTRAL (one word only).\n"
            f"Text: {state['message']}"
        ))
    ])
    sentiment = result.content.strip().upper()
    print(f"  [Node: analyze_sentiment] → {sentiment}")
    # Return ONLY the field(s) that this node updates
    return {"sentiment": sentiment}

def generate_response(state: CustomerState) -> dict:
    """Node 2: Generate a response tailored to the detected sentiment."""
    sentiment = state["sentiment"]
    message   = state["message"]

    # Customize tone based on sentiment from previous node
    tone = {
        "NEGATIVE": "Be very apologetic, empathetic, and offer immediate resolution.",
        "POSITIVE": "Be warm and appreciative. Match their positive energy.",
        "NEUTRAL":  "Be professional, helpful, and clear.",
    }.get(sentiment, "Be professional and helpful.")

    result = llm.invoke([
        HumanMessage(content=f"{tone}\nCustomer message: {message}\nWrite a reply:")
    ])
    print(f"  [Node: generate_response] → Response generated")
    return {"response": result.content}

# --- Step 3: Build the Graph ---
graph = StateGraph(CustomerState)

# Add nodes — each is a (name, function) pair
graph.add_node("analyze_sentiment", analyze_sentiment)
graph.add_node("generate_response", generate_response)

# Set entry point — where execution starts
graph.set_entry_point("analyze_sentiment")

# Add edges — define the flow
graph.add_edge("analyze_sentiment", "generate_response")  # always go here
graph.add_edge("generate_response", END)                   # then stop

# --- Step 4: Compile and Run ---
app = graph.compile()

test_messages = [
    "My order arrived damaged and nobody is responding to my emails!",
    "Just received my order — everything looks perfect, thank you!",
]

for msg in test_messages:
    print(f"\nInput: {msg}")
    result = app.invoke({"message": msg})
    print(f"Sentiment: {result['sentiment']}")
    print(f"Response: {result['response'][:150]}...")
    print("-" * 60)

---
## Section 4 — Conditional Edges: Branching

After a node runs, a **router function** reads the current state
and returns the **name of the next node** to execute.

```python
def router(state) -> str:
    if state["issue_type"] == "billing":
        return "billing_handler"
    else:
        return "general_handler"

graph.add_conditional_edges(
    "classifier",   # source node
    router,         # function that returns next node name
    {               # mapping: return value → node name
        "billing_handler": "billing_handler",
        "general_handler": "general_handler",
    }
)
```

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# ============================================================
# Conditional Routing: Route customer to the right department
# classify → (billing | technical | shipping | general) → END
# ============================================================

class SupportState(TypedDict):
    message: str
    department: str   # filled by classify node
    response: str     # filled by handler node

# --- Classifier node ---
def classify(state: SupportState) -> dict:
    """Classify which department should handle this issue."""
    result = llm.invoke([HumanMessage(content=(
        f"Classify this customer issue into ONE category: "
        f"billing, technical, shipping, or general.\n"
        f"Return ONLY the category word.\n"
        f"Issue: {state['message']}"
    ))])
    dept = result.content.strip().lower()
    print(f"  [classify] → {dept}")
    return {"department": dept}

# --- Department handler nodes ---
def handle_billing(state: SupportState) -> dict:
    r = llm.invoke([HumanMessage(content=f"[BILLING] Resolve: {state['message']}. Be brief.")])
    print("  [billing handler] → Response generated")
    return {"response": f"[Billing Team] {r.content}"}

def handle_technical(state: SupportState) -> dict:
    r = llm.invoke([HumanMessage(content=f"[TECH SUPPORT] Resolve: {state['message']}. Be brief.")])
    print("  [technical handler] → Response generated")
    return {"response": f"[Tech Support] {r.content}"}

def handle_shipping(state: SupportState) -> dict:
    r = llm.invoke([HumanMessage(content=f"[SHIPPING] Resolve: {state['message']}. Be brief.")])
    print("  [shipping handler] → Response generated")
    return {"response": f"[Shipping Team] {r.content}"}

def handle_general(state: SupportState) -> dict:
    r = llm.invoke([HumanMessage(content=f"[GENERAL SUPPORT] Resolve: {state['message']}. Be brief.")])
    print("  [general handler] → Response generated")
    return {"response": f"[General Support] {r.content}"}

# --- Router function ---
# Reads state and returns the NAME of the next node to execute
# Literal type hint documents the possible return values
def route_to_department(state: SupportState) -> Literal["billing", "technical", "shipping", "general"]:
    dept = state.get("department", "general")
    if "billing" in dept:   return "billing"
    if "tech" in dept:      return "technical"
    if "shipping" in dept:  return "shipping"
    return "general"

# --- Build graph ---
graph = StateGraph(SupportState)

graph.add_node("classify",  classify)
graph.add_node("billing",   handle_billing)
graph.add_node("technical", handle_technical)
graph.add_node("shipping",  handle_shipping)
graph.add_node("general",   handle_general)

graph.set_entry_point("classify")

# Conditional edge: after classify, call route_to_department to pick next node
graph.add_conditional_edges(
    "classify",              # from this node
    route_to_department,     # call this function to decide where to go
    {                        # map return values to node names
        "billing":   "billing",
        "technical": "technical",
        "shipping":  "shipping",
        "general":   "general",
    }
)

# All departments lead to END
for dept in ["billing", "technical", "shipping", "general"]:
    graph.add_edge(dept, END)

app = graph.compile()

# Test different messages — each should route to the correct department
test_cases = [
    "I was charged twice for my subscription last month!",
    "My app keeps crashing when I try to checkout.",
    "My package has been stuck at the warehouse for 2 weeks.",
]

print("Customer Support Routing System")
print("=" * 60)
for msg in test_cases:
    print(f"\nCustomer: {msg}")
    result = app.invoke({"message": msg})
    print(f"Response: {result['response'][:200]}")
    print("-" * 60)

---
## Section 5 — Loops: The Killer Feature

**This is the biggest reason to use LangGraph over LangChain chains.**

Loops let you build agents that:
- Keep trying until they get it right
- Research in multiple rounds
- Retry on failure
- Iterate until a quality threshold is met

```
START → research → evaluate ──(not enough)──→ research (loop!)
                       │
                  (sufficient)
                       │
                   synthesize → END
```

The loop is created simply by adding an edge that goes BACKWARDS:
```python
graph.add_edge("evaluate", "research")  # loop back!
```

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o", temperature=0.7)

# ============================================================
# Research Agent with Loop
# Keeps researching until it has enough information to answer
# ============================================================

class ResearchState(TypedDict):
    question: str             # the question to research
    findings: List[str]       # accumulated findings across iterations
    iteration: int            # how many research iterations done so far
    is_sufficient: bool       # flag: do we have enough info?
    final_answer: str         # synthesized final answer

# --- Research node: gather one piece of information ---
def research(state: ResearchState) -> dict:
    """Search for information about a specific aspect of the question."""
    question   = state["question"]
    iteration  = state.get("iteration", 0)
    findings   = state.get("findings", [])

    # Each iteration researches a DIFFERENT aspect
    aspects = [
        f"What is the basic definition and core concept of: {question}",
        f"What are the main real-world applications of: {question}",
        f"What are the key limitations or challenges of: {question}",
    ]
    aspect = aspects[min(iteration, len(aspects) - 1)]

    result = llm.invoke([
        HumanMessage(content=f"Research this in 2-3 sentences: {aspect}")
    ])

    new_finding = f"[Iteration {iteration + 1}]: {result.content}"
    print(f"  [research] Iteration {iteration + 1}: researched '{aspect[:50]}...}'")

    return {
        "findings": findings + [new_finding],  # append to existing findings
        "iteration": iteration + 1
    }

# --- Evaluate node: decide if we have enough info ---
def evaluate(state: ResearchState) -> dict:
    """Check if we have enough findings to produce a good answer."""
    findings  = state["findings"]
    iteration = state["iteration"]

    # Stop when we have 3 findings OR hit the 5-iteration safety limit
    is_sufficient = len(findings) >= 3 or iteration >= 5
    print(f"  [evaluate] {len(findings)} findings, sufficient={is_sufficient}")
    return {"is_sufficient": is_sufficient}

# --- Synthesize node: combine findings into final answer ---
def synthesize(state: ResearchState) -> dict:
    """Combine all research findings into a comprehensive answer."""
    combined = "\n".join(state["findings"])
    result = llm.invoke([
        HumanMessage(content=(
            f"Based on these research findings, write a comprehensive answer to: '{state['question']}'\n\n"
            f"Findings:\n{combined}"
        ))
    ])
    print("  [synthesize] Final answer generated")
    return {"final_answer": result.content}

# --- Router: loop back to research OR move to synthesize ---
def should_continue(state: ResearchState) -> str:
    """Return 'research' to loop, or 'synthesize' to finish."""
    return "synthesize" if state.get("is_sufficient", False) else "research"

# --- Build graph with loop ---
graph = StateGraph(ResearchState)

graph.add_node("research",   research)
graph.add_node("evaluate",   evaluate)
graph.add_node("synthesize", synthesize)

graph.set_entry_point("research")

graph.add_edge("research", "evaluate")  # always evaluate after each research

# The LOOP: after evaluate, go back to research OR forward to synthesize
graph.add_conditional_edges(
    "evaluate",
    should_continue,
    {
        "research":   "research",   # ← THIS IS THE LOOP (edge goes backward)
        "synthesize": "synthesize"  # forward to finish
    }
)
graph.add_edge("synthesize", END)

app = graph.compile()

print("🔬 Research Agent (with loop)")
print("=" * 60)

result = app.invoke({
    "question":      "What is Retrieval-Augmented Generation (RAG)?",
    "findings":      [],
    "iteration":     0,
    "is_sufficient": False,
    "final_answer":  ""
})

print(f"\nTotal iterations: {result['iteration']}")
print(f"\nFinal Answer:\n{result['final_answer']}")

---
## Section 6 — Persistence: Save and Resume Graph State

LangGraph can **checkpoint** the state at every node.
This lets you:
- Resume a workflow after a crash
- Pause for human input and resume
- Run long workflows over multiple sessions
- Replay from any checkpoint for debugging

```python
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()              # in-memory (for dev)
app = graph.compile(checkpointer=checkpointer)

# thread_id groups checkpoints for the same conversation
config = {"configurable": {"thread_id": "session-123"}}
app.invoke(input_data, config=config)
```

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatOpenAI(model="gpt-4o", temperature=0.3)

# ============================================================
# Persistent Chatbot — remembers conversation across calls
# Each .invoke() continues where the previous one left off
# ============================================================

class ChatState(TypedDict):
    messages: List  # list of all messages (SystemMessage/HumanMessage/AIMessage)

def chat_node(state: ChatState) -> dict:
    """Call the LLM with the full conversation history."""
    response = llm.invoke(state["messages"])
    # Append AI response to the message list
    return {"messages": state["messages"] + [AIMessage(content=response.content)]}

graph = StateGraph(ChatState)
graph.add_node("chat", chat_node)
graph.set_entry_point("chat")
graph.add_edge("chat", END)

# MemorySaver: stores checkpoints in memory (use SqliteSaver for disk persistence)
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

# thread_id identifies this conversation — same ID = continues same conversation
config = {"configurable": {"thread_id": "tutor-session-001"}}

SYSTEM = [SystemMessage(content="You are a concise Python tutor.")]

def persistent_chat(user_input: str):
    """Each call continues the same conversation thread."""
    # Get current state to append to
    current = app.get_state(config)
    if current.values:  # if conversation exists, append to it
        existing = current.values["messages"]
        messages = existing + [HumanMessage(content=user_input)]
    else:               # first turn: start with system + user message
        messages = SYSTEM + [HumanMessage(content=user_input)]

    result = app.invoke({"messages": messages}, config=config)
    return result["messages"][-1].content  # return last AI message

print("Persistent Chat (state saved between calls)")
print("=" * 60)

# Each call continues the same conversation — state is checkpointed
q1 = persistent_chat("What is a Python generator?")
print(f"Turn 1: {q1[:200]}")
print()

q2 = persistent_chat("Can you show a simple code example?")  # AI remembers context
print(f"Turn 2: {q2[:200]}")

---
## Section 7 — Multi-Agent: Supervisor Pattern

In complex tasks, instead of one agent doing everything, you create **specialist agents**
and a **supervisor** that coordinates them.

```
User request
      │
      ▼
  SUPERVISOR ──────────────────────┐
      │                            │
      ├──→ Research Agent          │
      ├──→ Writing Agent    ───────┘ (all report back to supervisor)
      └──→ QA Agent
             │
           Done → END
```

**Why this pattern?**
- Each agent is focused and does one thing well
- Supervisor decides the order and when to stop
- Easy to add/remove specialist agents
- **Java analogy:** Microservices with an API Gateway/Orchestrator

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model="gpt-4o", temperature=0.3)

# ============================================================
# Multi-Agent Blog Writer
# Supervisor → Research Agent → Writing Agent → QA Agent → Done
# ============================================================

class BlogState(TypedDict):
    topic: str                # input topic
    research_notes: str       # filled by researcher
    draft: str                # filled by writer
    qa_feedback: str          # filled by qa agent
    final_post: str           # finalized content
    completed_steps: List[str]  # tracks which agents have run
    next_step: str            # supervisor's routing decision

# --- Supervisor ---
def supervisor(state: BlogState) -> dict:
    """Decide which specialist agent to run next."""
    done = state.get("completed_steps", [])

    # Simple sequential workflow — in production use LLM to decide dynamically
    if "research" not in done:  next_step = "research"
    elif "write" not in done:   next_step = "write"
    elif "qa" not in done:      next_step = "qa"
    else:                       next_step = "done"

    print(f"  [supervisor] Routing to: {next_step}")
    return {"next_step": next_step}

# --- Specialist Agents ---
def research_agent(state: BlogState) -> dict:
    """Gathers key facts and insights about the topic."""
    result = llm.invoke([
        SystemMessage(content="You are a research specialist. Be factual and comprehensive."),
        HumanMessage(content=f"Research '{state['topic']}'. Provide 4 key points with brief explanations.")
    ])
    done = state.get("completed_steps", []) + ["research"]
    print(f"  [research_agent] Notes gathered ({len(result.content)} chars)")
    return {"research_notes": result.content, "completed_steps": done}

def writing_agent(state: BlogState) -> dict:
    """Writes a blog post using the research notes."""
    result = llm.invoke([
        SystemMessage(content="You are a professional blog writer. Write engaging, clear content."),
        HumanMessage(content=(
            f"Write a 200-word blog post about '{state['topic']}'.\n"
            f"Use these research notes:\n{state['research_notes']}"
        ))
    ])
    done = state.get("completed_steps", []) + ["write"]
    print(f"  [writing_agent] Draft written ({len(result.content)} chars)")
    return {"draft": result.content, "completed_steps": done}

def qa_agent(state: BlogState) -> dict:
    """Reviews the draft for quality and accuracy."""
    result = llm.invoke([
        SystemMessage(content="You are a content QA reviewer. Check for accuracy, clarity, and completeness."),
        HumanMessage(content=f"Review this blog post. Give a short verdict and one improvement suggestion:\n{state['draft']}")
    ])
    done = state.get("completed_steps", []) + ["qa"]
    # In production: use QA feedback to trigger a revision loop if quality < threshold
    print(f"  [qa_agent] Review complete")
    return {
        "qa_feedback": result.content,
        "final_post": state["draft"],  # approved
        "completed_steps": done
    }

# --- Router for supervisor ---
def route_from_supervisor(state: BlogState) -> str:
    return state.get("next_step", "done")

# --- Build multi-agent graph ---
graph = StateGraph(BlogState)

graph.add_node("supervisor", supervisor)
graph.add_node("research",   research_agent)
graph.add_node("write",      writing_agent)
graph.add_node("qa",         qa_agent)

graph.set_entry_point("supervisor")

# Supervisor routes to the correct specialist
graph.add_conditional_edges(
    "supervisor", route_from_supervisor,
    {"research": "research", "write": "write", "qa": "qa", "done": END}
)

# ALL specialists report back to supervisor after finishing
for agent in ["research", "write", "qa"]:
    graph.add_edge(agent, "supervisor")

app = graph.compile()

print("📝 Multi-Agent Blog Writer")
print("=" * 60)

result = app.invoke({
    "topic": "How LangGraph enables stateful AI workflows",
    "research_notes": "", "draft": "", "qa_feedback": "",
    "final_post": "", "completed_steps": [], "next_step": ""
})

print(f"\n📰 Final Blog Post:\n{result['final_post']}")
print(f"\n✅ QA Verdict:\n{result['qa_feedback'][:200]}...")

---
## Section 8 — Human-in-the-Loop

Some actions need **human approval** before proceeding — sending emails,
processing refunds, publishing content.

LangGraph supports this via **interrupt_before** — the graph PAUSES before
a specified node, saves state, and waits for you to resume it.

```python
# Pause before 'send_email' node runs
app = graph.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["send_email"]
)

# First call: runs until just BEFORE send_email, then pauses
app.invoke(input_data, config=config)

# Human reviews, then resumes with updated state if needed
app.update_state(config, {"approved": True})
app.invoke(None, config=config)  # resume from checkpoint
```

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o", temperature=0.3)

# ============================================================
# Human-in-the-Loop: Email Draft Approval Workflow
# 1. AI drafts an email
# 2. PAUSE — human reviews and optionally edits
# 3. RESUME — send the approved email
# ============================================================

class EmailWorkflowState(TypedDict):
    complaint: str          # customer complaint
    draft: str              # AI-drafted email
    human_notes: str        # human's edits/feedback
    final_email: str        # approved and finalized email
    approved: bool          # whether human approved

def draft_email(state: EmailWorkflowState) -> dict:
    """AI drafts the email response."""
    result = llm.invoke([
        HumanMessage(content=(
            f"Draft a professional customer service response to:\n{state['complaint']}\n"
            f"Be empathetic, clear, and offer a concrete resolution."
        ))
    ])
    print("  [draft_email] Draft created")
    print(f"\n  Draft Preview: {result.content[:200]}...")
    return {"draft": result.content}

def send_email(state: EmailWorkflowState) -> dict:
    """Send the email (in production: call your email API here)."""
    # Use human-edited version if provided, otherwise use original draft
    final = state["human_notes"] if state.get("human_notes") else state["draft"]
    print("  [send_email] Email sent!")
    return {"final_email": final}

graph = StateGraph(EmailWorkflowState)
graph.add_node("draft_email", draft_email)
graph.add_node("send_email",  send_email)

graph.set_entry_point("draft_email")
graph.add_edge("draft_email", "send_email")
graph.add_edge("send_email",  END)

checkpointer = MemorySaver()

# interrupt_before=["send_email"] means: PAUSE before send_email executes
# The graph runs draft_email, then STOPS — state is saved
app = graph.compile(
    checkpointer=checkpointer,
    interrupt_before=["send_email"]  # pause here for human review
)

config = {"configurable": {"thread_id": "complaint-review-001"}}

complaint = "I ordered a laptop worth ₹75,000 two weeks ago but received the wrong model."

print("Email Approval Workflow")
print("=" * 60)
print(f"Complaint: {complaint}")
print()

# PHASE 1: Run until interrupt (draft_email runs, then pauses before send_email)
app.invoke(
    {"complaint": complaint, "draft": "", "human_notes": "",
     "final_email": "", "approved": False},
    config=config
)

print("\n⏸️  PAUSED: Waiting for human review...")

# PHASE 2: Simulate human reviewing and optionally editing the draft
# In production: webhook/API call updates state, then resumes
human_edit = "Also include a 10% discount coupon for the inconvenience."
print(f"Human adds: {human_edit}")

# Update state with human's input before resuming
app.update_state(config, {"human_notes": human_edit, "approved": True})

# PHASE 3: Resume from checkpoint — send_email node now executes
print("▶️  Resuming workflow...")
final = app.invoke(None, config=config)  # None = resume from where we paused

print(f"\n✅ Final email sent:")
print(final["final_email"][:400])

---
## ✅ LangGraph Summary

| Concept | API | Notes |
|---|---|---|
| **State definition** | `class S(TypedDict)` | All nodes share this |
| **Create graph** | `StateGraph(S)` | Pass your state type |
| **Add node** | `graph.add_node("name", fn)` | fn receives state, returns dict |
| **Set start** | `graph.set_entry_point("name")` | First node to run |
| **Normal edge** | `graph.add_edge("A", "B")` | Always A → B |
| **Conditional edge** | `graph.add_conditional_edges("A", router, map)` | Router returns node name |
| **Loop** | `graph.add_edge("B", "A")` | Just add a backward edge |
| **Compile** | `app = graph.compile()` | Makes it runnable |
| **Run** | `app.invoke(state_dict)` | Returns final state |
| **Persistence** | `compile(checkpointer=MemorySaver())` | Saves state between calls |
| **Human pause** | `compile(interrupt_before=["node"])` | Pauses before that node |
| **Resume** | `app.invoke(None, config=config)` | Continues from checkpoint |

---

## When to Use What

```
Simple pipeline (A→B→C)?         → LangChain LCEL chain
Need loops?                       → LangGraph
Complex routing (if/else)?        → LangGraph conditional edges
Multiple AI agents collaborating? → LangGraph multi-agent
Need human approval step?         → LangGraph interrupt_before
Long-running workflow with state? → LangGraph + checkpointing
```

## 🚀 Next
See `07_ai_frameworks_comparison.md` for how LangChain/LangGraph
compare to LlamaIndex, Haystack, AutoGen, CrewAI, and others.